# Convolutional Neural Networks — Theory (2 hours)

**Course:** ML, Deep Learning & Computer Vision — Phase 4  
**Prerequisites:** Phase 3 (neurons, backprop, PyTorch)  

---

| Time | Topic |
|------|-------|
| 0:00–0:20 | Why fully-connected fails for images |
| 0:20–0:50 | Convolutions — the core idea |
| 0:50–1:10 | Pooling, stride, padding |
| 1:10–1:35 | Classic architectures (LeNet → ResNet) |
| 1:35–1:55 | Receptive field, parameter counting, 1×1 convolutions |
| 1:55–2:00 | Summary |

By the end, you'll understand **what** a convolution computes, **why** CNNs work for images, and **how** the key architectures evolved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

np.random.seed(42)
np.set_printoptions(precision=2, suppress=True)

---
# Chapter 1 — Why Fully-Connected Fails for Images

## 1.1 The parameter explosion

A fully-connected layer connecting every input pixel to every neuron:

| Image size | Flattened | Hidden layer (256) | Parameters |
|-----------|-----------|-------------------|------------|
| 28×28 (MNIST) | 784 | 256 | 200,960 |
| 224×224×3 (ImageNet) | 150,528 | 256 | **38.5 million** |
| 1024×1024×3 (HD) | 3,145,728 | 256 | **805 million** |

Just the **first layer** of a fully-connected network on an HD image would have 805M parameters. That's more than GPT-2.

## 1.2 Three problems with FC for images

1. **Too many parameters** → overfits immediately, impossible to train
2. **No spatial structure** → flattening destroys the 2D arrangement. A pixel at (0,0) and (100,100) are treated the same as adjacent pixels
3. **No translation invariance** → a cat in the top-left and a cat in the bottom-right require completely different weights to detect

CNNs solve all three problems with one idea: **local connectivity + weight sharing**.

In [ ]:
# Demonstrate the parameter explosion
image_sizes = [
    ("MNIST", 28, 28, 1),
    ("CIFAR-10", 32, 32, 3),
    ("ImageNet", 224, 224, 3),
    ("HD photo", 1024, 1024, 3),
]

hidden = 256
print(f"{'Dataset':15s} {'Image size':15s} {'Flattened':>10s} {'FC params':>12s}")
print("-" * 55)
for name, h, w, c in image_sizes:
    flat = h * w * c
    params = flat * hidden + hidden  # weights + bias
    print(f"{name:15s} {h}×{w}×{c:<8s} {flat:>10,} {params:>12,}")

print(f"\nA single Conv layer with 64 filters of 3×3 on ANY size image:")
conv_params = 3 * 3 * 3 * 64 + 64  # kernel_h × kernel_w × in_channels × out_channels + bias
print(f"  Parameters: {conv_params:,}  ← same regardless of image size!")
print(f"  That's {(224*224*3*256) / conv_params:.0f}× fewer than FC on ImageNet.")

---
# Chapter 2 — Convolutions: The Core Idea

## 2.1 What is a convolution?

A convolution slides a small **filter** (also called a **kernel**) across the image, computing a dot product at each position.

```
Input image (5×5)          Filter (3×3)          Output (3×3)
┌─────────────┐           ┌───────┐           ┌─────────┐
│ 1 0 1 0 1   │           │ 1 0 1 │           │ 4 3 4   │
│ 0 1 0 1 0   │    *      │ 0 1 0 │    =      │ 2 4 3   │
│ 1 0 1 0 1   │           │ 1 0 1 │           │ 4 3 4   │
│ 0 1 0 1 0   │           └───────┘           └─────────┘
│ 1 0 1 0 1   │
└─────────────┘
```

At each position, the filter is placed on top of the image, element-wise multiplied, and summed → one output value.

**Key insight:** The same filter is used at every position. This is **weight sharing** — the network learns to detect a feature (edge, corner, texture) *regardless of where it appears*.

In [ ]:
# Implement 2D convolution from scratch
def conv2d(image, kernel, stride=1, padding=0):
    """
    2D convolution (single channel).
    image:  (H, W)
    kernel: (kH, kW)
    returns: (out_H, out_W)
    """
    if padding > 0:
        image = np.pad(image, padding, mode='constant', constant_values=0)
    
    H, W = image.shape
    kH, kW = kernel.shape
    out_H = (H - kH) // stride + 1
    out_W = (W - kW) // stride + 1
    
    output = np.zeros((out_H, out_W))
    for i in range(out_H):
        for j in range(out_W):
            # Extract the patch under the kernel
            patch = image[i*stride:i*stride+kH, j*stride:j*stride+kW]
            # Element-wise multiply and sum
            output[i, j] = np.sum(patch * kernel)
    
    return output

# Step-by-step demo
image = np.array([
    [1, 0, 1, 0, 1],
    [0, 1, 0, 1, 0],
    [1, 0, 1, 0, 1],
    [0, 1, 0, 1, 0],
    [1, 0, 1, 0, 1],
], dtype=float)

kernel = np.array([
    [1, 0, 1],
    [0, 1, 0],
    [1, 0, 1],
], dtype=float)

output = conv2d(image, kernel)
print("Image (5×5):")
print(image)
print(f"\nKernel (3×3):")
print(kernel)
print(f"\nOutput (3×3):")
print(output)

# Show the first computation in detail
patch = image[0:3, 0:3]
print(f"\n--- First output value ---")
print(f"Patch (top-left 3×3):")
print(patch)
print(f"Element-wise multiply:")
print(patch * kernel)
print(f"Sum = {np.sum(patch * kernel):.0f}")

## 2.2 Kernels as feature detectors

Different kernels detect different features:

| Kernel | What it detects |
|--------|----------------|
| `[[-1,0,1],[-1,0,1],[-1,0,1]]` | Vertical edges |
| `[[-1,-1,-1],[0,0,0],[1,1,1]]` | Horizontal edges |
| `[[0,-1,0],[-1,4,-1],[0,-1,0]]` | Edges in all directions (Laplacian) |
| `[[1,1,1],[1,1,1],[1,1,1]] / 9` | Blur (average) |
| `[[0,-1,0],[-1,5,-1],[0,-1,0]]` | Sharpen |

In a CNN, the network **learns** these kernels from data. It discovers the features that matter for the task.

In [ ]:
# Apply classic kernels to a real-looking image
# Create a simple test image with edges
img = np.zeros((64, 64))
img[10:50, 10:50] = 1.0           # white square
img[20:40, 20:40] = 0.5           # gray inner square
img[25:35, 25:35] = 1.0           # white center
# Add diagonal line
for i in range(64):
    if 0 <= i < 64:
        img[i, min(i, 63)] = 1.0

kernels = {
    "Original": None,
    "Vertical edges": np.array([[-1,0,1],[-1,0,1],[-1,0,1]], dtype=float),
    "Horizontal edges": np.array([[-1,-1,-1],[0,0,0],[1,1,1]], dtype=float),
    "Laplacian (all edges)": np.array([[0,-1,0],[-1,4,-1],[0,-1,0]], dtype=float),
    "Blur (3×3 average)": np.ones((3,3)) / 9,
    "Sharpen": np.array([[0,-1,0],[-1,5,-1],[0,-1,0]], dtype=float),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (name, kernel) in zip(axes.flat, kernels.items()):
    if kernel is None:
        ax.imshow(img, cmap='gray')
    else:
        out = conv2d(img, kernel)
        ax.imshow(out, cmap='RdBu_r' if 'edge' in name.lower() or 'Laplacian' in name else 'gray')
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.axis('off')

plt.suptitle('Hand-crafted kernels as feature detectors', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("In a CNN, the network LEARNS these kernels from data.")
print("Early layers learn simple features (edges). Deeper layers learn complex ones (eyes, wheels).")

## 2.3 Multi-channel convolutions (the real version)

Real images have **3 channels** (RGB). A convolution kernel is actually 3D:

$$\text{Output}[i,j] = \sum_{c=0}^{C_{in}-1} \sum_{m=0}^{k-1} \sum_{n=0}^{k-1} \text{Input}[c, i+m, j+n] \cdot \text{Kernel}[c, m, n] + b$$

**Shape rules:**
- Input: `(C_in, H, W)` — e.g., (3, 224, 224) for an RGB image
- One kernel: `(C_in, kH, kW)` — e.g., (3, 3, 3) = 27 weights
- One kernel produces: `(1, H_out, W_out)` — one feature map
- N kernels produce: `(N, H_out, W_out)` — N feature maps

**Key insight:** Each kernel spans ALL input channels but produces a single output channel. Multiple kernels → multiple feature maps.

In [ ]:
# Multi-channel convolution from scratch
def conv2d_multichannel(image, kernels, bias=None, stride=1, padding=0):
    """
    image:   (C_in, H, W)
    kernels: (C_out, C_in, kH, kW)
    bias:    (C_out,) or None
    returns: (C_out, H_out, W_out)
    """
    C_out, C_in, kH, kW = kernels.shape
    
    # Pad each channel
    if padding > 0:
        image = np.pad(image, ((0,0),(padding,padding),(padding,padding)), mode='constant')
    
    _, H, W = image.shape
    out_H = (H - kH) // stride + 1
    out_W = (W - kW) // stride + 1
    output = np.zeros((C_out, out_H, out_W))
    
    for f in range(C_out):        # for each output filter
        for i in range(out_H):
            for j in range(out_W):
                # Sum across ALL input channels
                patch = image[:, i*stride:i*stride+kH, j*stride:j*stride+kW]
                output[f, i, j] = np.sum(patch * kernels[f]) + (bias[f] if bias is not None else 0)
    
    return output

# Demo: RGB image → 4 feature maps
rgb_image = np.random.randn(3, 8, 8)  # (3, 8, 8)
kernels = np.random.randn(4, 3, 3, 3)  # 4 filters, each (3, 3, 3)
bias = np.zeros(4)

output = conv2d_multichannel(rgb_image, kernels, bias, padding=1)
print(f"Input shape:   {rgb_image.shape}  (3 channels, 8×8)")
print(f"Kernels shape: {kernels.shape}  (4 filters, each 3×3×3)")
print(f"Output shape:  {output.shape}  (4 feature maps, 8×8)")
print(f"\nParameters: {kernels.size + bias.size} = 4 × (3×3×3) + 4 = {4*27} + 4 = {4*27+4}")
print(f"This is the SAME regardless of image size — a 1000×1000 image uses the same 112 params!")

---
# Chapter 3 — Stride, Padding & Pooling

## 3.1 Output size formula

$$H_{out} = \left\lfloor \frac{H_{in} + 2P - k}{S} \right\rfloor + 1$$

where:
- $H_{in}$ = input height
- $P$ = padding (zeros added around the border)
- $k$ = kernel size
- $S$ = stride (step size of the sliding window)

## 3.2 Padding — preserving spatial dimensions

Without padding, each conv layer shrinks the feature map. With `padding = k//2` (called **same padding**), the output has the same spatial size as the input.

## 3.3 Stride — downsampling

Stride > 1 moves the kernel more than one pixel at a time, reducing the output size. Stride 2 halves each dimension.

In [ ]:
# Output size calculator
def output_size(H_in, kernel, stride=1, padding=0):
    return (H_in + 2*padding - kernel) // stride + 1

print("=== Output size examples ===")
print(f"{'Input':>8s} {'Kernel':>8s} {'Stride':>8s} {'Padding':>8s} → {'Output':>8s}")
print("-" * 50)
configs = [
    (32, 3, 1, 0),   # no padding
    (32, 3, 1, 1),   # same padding → output = input
    (32, 3, 2, 1),   # stride 2 → halved
    (32, 5, 1, 2),   # 5×5 kernel, same padding
    (224, 7, 2, 3),  # first conv in ResNet
    (224, 3, 1, 1),  # standard conv block
    (112, 3, 2, 1),  # downsample
]
for h, k, s, p in configs:
    out = output_size(h, k, s, p)
    note = "← same!" if out == h else ("← halved" if out == h//2 else "")
    print(f"{h:>8d} {k:>8d} {s:>8d} {p:>8d} → {out:>8d}  {note}")

print("\nRule of thumb: kernel=3, stride=1, padding=1 → same size (most common)")
print("              kernel=3, stride=2, padding=1 → half size (downsampling)")

## 3.4 Pooling — spatial downsampling

**Max pooling** takes the maximum value in each window. It:
- Reduces spatial dimensions (typically by 2×)
- Provides a small amount of translation invariance
- Has **no learnable parameters**

**Global average pooling (GAP)** averages over the entire spatial dimension, producing a single value per channel. Used at the end of modern networks instead of flattening.

In [ ]:
# Max pooling from scratch
def max_pool_2d(x, pool_size=2, stride=2):
    """x shape: (C, H, W) → (C, H//2, W//2)"""
    C, H, W = x.shape
    out_H = H // stride
    out_W = W // stride
    output = np.zeros((C, out_H, out_W))
    
    for c in range(C):
        for i in range(out_H):
            for j in range(out_W):
                patch = x[c, i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
                output[c, i, j] = np.max(patch)
    
    return output

# Demo
feature_map = np.random.randn(1, 8, 8).round(1)
pooled = max_pool_2d(feature_map)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(feature_map[0], cmap='viridis')
axes[0].set_title(f'Before pooling {feature_map.shape[1:]}', fontweight='bold')
for i in range(8):
    for j in range(8):
        axes[0].text(j, i, f'{feature_map[0,i,j]:.1f}', ha='center', va='center', fontsize=7)

axes[1].imshow(pooled[0], cmap='viridis')
axes[1].set_title(f'After 2×2 max pooling {pooled.shape[1:]}', fontweight='bold')
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, f'{pooled[0,i,j]:.1f}', ha='center', va='center', fontsize=9)

# Draw grid on the before image showing 2×2 blocks
for i in range(0, 9, 2):
    axes[0].axhline(i - 0.5, color='red', linewidth=1.5, alpha=0.5)
    axes[0].axvline(i - 0.5, color='red', linewidth=1.5, alpha=0.5)

plt.suptitle('Max pooling: take the maximum from each 2×2 block', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Input:  {feature_map.shape} → Output: {pooled.shape}")
print(f"Spatial dimensions halved. No learnable parameters.")

---
# Chapter 4 — A CNN Layer-by-Layer

A typical CNN block:

```
Input → [Conv → BatchNorm → ReLU → Pool] × N → Global Avg Pool → FC → Output
```

Each conv block:
1. **Conv** — detect features using learned kernels
2. **BatchNorm** — normalise activations for stable training
3. **ReLU** — introduce non-linearity
4. **Pool** — reduce spatial dimensions

As we go deeper:
- **Spatial size decreases** (pooling/stride reduces H, W)
- **Number of channels increases** (more feature maps detect more complex features)
- **Receptive field grows** (each neuron sees a larger portion of the original image)

In [ ]:
# Trace shapes through a typical CNN
def trace_cnn(input_shape):
    """Trace tensor shapes through a VGG-like network."""
    C, H, W = input_shape
    layers = [
        ("Input",           C, H, W, 0),
        ("Conv 3×3, 64",    64, H, W, 3*3*C*64+64),
        ("Conv 3×3, 64",    64, H, W, 3*3*64*64+64),
        ("MaxPool 2×2",     64, H//2, W//2, 0),
        ("Conv 3×3, 128",   128, H//2, W//2, 3*3*64*128+128),
        ("Conv 3×3, 128",   128, H//2, W//2, 3*3*128*128+128),
        ("MaxPool 2×2",     128, H//4, W//4, 0),
        ("Conv 3×3, 256",   256, H//4, W//4, 3*3*128*256+256),
        ("Conv 3×3, 256",   256, H//4, W//4, 3*3*256*256+256),
        ("MaxPool 2×2",     256, H//8, W//8, 0),
        ("Global Avg Pool", 256, 1, 1, 0),
        ("FC → 10 classes", 10, 1, 1, 256*10+10),
    ]
    
    total_params = 0
    print(f"{'Layer':20s} {'Output shape':>18s} {'Feature map':>14s} {'Params':>10s}")
    print("-" * 65)
    for name, c, h, w, params in layers:
        total_params += params
        fm_size = c * h * w
        p_str = f"{params:,}" if params > 0 else "-"
        print(f"{name:20s} ({c:>3d}, {h:>3d}, {w:>3d}) {fm_size:>14,} {p_str:>10s}")
    print(f"{'':20s} {'':>18s} {'':>14s} {'─'*10}")
    print(f"{'TOTAL':20s} {'':>18s} {'':>14s} {total_params:>10,}")

print("=== CNN on 32×32 image (CIFAR-10) ===")
trace_cnn((3, 32, 32))

print("\n\n=== Same CNN on 224×224 image (ImageNet) ===")
trace_cnn((3, 224, 224))

print("\nNotice: SAME number of parameters regardless of image size!")
print("The feature maps get bigger but the kernels stay the same.")

## 4.1 What does each layer learn?

This is one of the most important insights in deep learning:

| Depth | What it detects | Receptive field |
|-------|----------------|----------------|
| **Layer 1** | Edges, colours, simple gradients | 3×3 pixels |
| **Layer 2** | Corners, textures, simple shapes | 5×5 pixels |
| **Layer 3–4** | Parts of objects (eyes, wheels, windows) | 10–20 pixels |
| **Layer 5–7** | Whole objects, complex patterns | 50–100 pixels |
| **Final layers** | Scene-level understanding | Entire image |

The network automatically learns a hierarchy from simple → complex features. This is why deep learning works — you don't engineer features, the network discovers them.

---
# Chapter 5 — Architecture Evolution: LeNet to ResNet

Each architecture solved a specific problem of its predecessor.

## 5.1 LeNet-5 (1998) — the original CNN

- **Creator:** Yann LeCun
- **Task:** Handwritten digit recognition (MNIST)
- **Architecture:** Conv → Pool → Conv → Pool → FC → FC → Output
- **Key idea:** Convolutions work for image classification
- **Parameters:** ~60K
- **Limitations:** Too small for complex images, used sigmoid/tanh

## 5.2 AlexNet (2012) — the deep learning revolution

- **Creator:** Alex Krizhevsky, Ilya Sutskever, Geoffrey Hinton
- **Task:** ImageNet (1000 classes, 1.2M images)
- **Key innovations:**
  - **ReLU** instead of sigmoid → faster training, no vanishing gradient
  - **Dropout** → regularisation for large networks
  - **GPU training** → first large CNN trained on GPUs
- **Parameters:** ~60M
- **Impact:** Won ImageNet 2012 by a huge margin, launched the deep learning era

## 5.3 VGGNet (2014) — depth matters

- **Key insight:** Use ONLY 3×3 convolutions, stacked deeply (16–19 layers)
- **Why 3×3?** Two stacked 3×3 convs have the same receptive field as one 5×5 conv, but fewer parameters and more non-linearity
- **Parameters:** ~138M (most in the FC layers)
- **Lesson:** deeper = better, but only up to a point...

In [ ]:
# Why 3×3 is better than 5×5
print("=== Why VGG uses 3×3 instead of 5×5 ===")
print("\nTwo stacked 3×3 convs = same receptive field as one 5×5 conv")
print("")

# Parameters comparison (C input channels, C output channels)
C = 256  # typical channel count
params_5x5 = 5 * 5 * C * C
params_3x3_stacked = 2 * (3 * 3 * C * C)

print(f"One 5×5 conv:          {params_5x5:>12,} params")
print(f"Two stacked 3×3 convs: {params_3x3_stacked:>12,} params")
print(f"Saving:                {params_5x5 - params_3x3_stacked:>12,} ({(1-params_3x3_stacked/params_5x5)*100:.0f}% fewer)")
print(f"\nPlus: two 3×3 convs have TWO ReLU activations (more non-linearity)")
print(f"      one 5×5 conv has only ONE ReLU")
print(f"\nMore non-linearity + fewer parameters = better model.")
print(f"This is why modern architectures almost exclusively use 3×3 kernels.")

## 5.4 ResNet (2015) — skip connections change everything

**The problem:** deeper networks should be better, but in practice very deep networks (50+ layers) performed *worse* than shallow ones. Not because of overfitting — even training loss was higher!

**The cause:** The **degradation problem**. In a very deep network, gradients vanish during backprop, making early layers nearly impossible to train.

**The solution:** **Skip connections** (residual connections). Instead of learning $H(x)$, learn the residual $F(x) = H(x) - x$:

$$\text{output} = F(x) + x$$

```
    x ──────────────────────┐
    │                       │ (skip connection)
    ├── Conv → BN → ReLU   │
    ├── Conv → BN          │
    │                       │
    └── + ← ────────────────┘
        │
       ReLU
        │
      output
```

**Why it works:**
1. **Gradient highway:** During backprop, the gradient flows directly through the skip connection (derivative of $x$ is 1). No vanishing gradient!
2. **Easy to learn identity:** If a layer isn't useful, it can learn $F(x) = 0$, making the block just pass the input through. Deeper networks are never worse than shallower ones.
3. **Residual learning is easier:** Learning a small correction $F(x)$ is easier than learning the full mapping $H(x)$.

In [ ]:
# ResNet block — the skip connection
def residual_block_forward(x, W1, b1, W2, b2):
    """
    Simple residual block (dense version for illustration).
    In practice, these use Conv instead of Dense.
    """
    # Main path
    z1 = x @ W1 + b1
    a1 = np.maximum(0, z1)     # ReLU
    z2 = a1 @ W2 + b2
    
    # Skip connection: add input directly to output
    output = z2 + x             # ← THIS is the key line!
    output = np.maximum(0, output)  # ReLU after addition
    
    return output

# Demo: show that gradients flow through skip connections
print("=== Gradient flow: plain vs residual ===")
print("\nPlain network (no skip): gradient must flow through ALL layers")
print("  After 50 layers with sigmoid: gradient ≈ 0.25^50 ≈ 0")
print("  After 50 layers with ReLU: gradient can still vanish if neurons die")
print("\nResidual network (with skip): gradient has a HIGHWAY")
print("  d(F(x) + x)/dx = dF(x)/dx + 1")
print("                              ^^^")
print("  This '+1' means the gradient is ALWAYS at least 1.")
print("  Even if the layers learn nothing (F(x)=0), gradient = 1.")
print("  This is why ResNets can have 152+ layers and still train.")

# Architecture comparison
print("\n=== Architecture comparison ===")
archs = [
    ("LeNet-5",  1998, 5,   "~60K",    "26.0%", "Started it all"),
    ("AlexNet",  2012, 8,   "~60M",    "16.4%", "ReLU, dropout, GPU"),
    ("VGG-16",   2014, 16,  "~138M",   "7.3%",  "Only 3×3 convs"),
    ("GoogLeNet",2014, 22,  "~6.8M",   "6.7%",  "Inception modules"),
    ("ResNet-152",2015,152, "~60M",    "3.6%",  "Skip connections"),
]
print(f"{'Name':12s} {'Year':>5s} {'Depth':>6s} {'Params':>8s} {'Top-5 err':>10s} {'Key innovation'}")
print("-" * 72)
for name, year, depth, params, err, innovation in archs:
    print(f"{name:12s} {year:>5d} {depth:>6d} {params:>8s} {err:>10s} {innovation}")

print("\nIn 3 years: error dropped from 26% to 3.6% (human performance ≈ 5.1%)")
print("ResNet surpassed human performance on ImageNet!")

---
# Chapter 6 — Receptive Field & Parameter Counting

## 6.1 Receptive field

The **receptive field** of a neuron is the region of the input image that influences its value.

- After one 3×3 conv: each neuron sees a 3×3 patch
- After two 3×3 convs: each neuron sees a 5×5 patch
- After three 3×3 convs: each neuron sees a 7×7 patch
- After 2×2 pooling: receptive field doubles

Formula for N stacked 3×3 convs (no pooling): receptive field = $2N + 1$

## 6.2 1×1 Convolutions — the dimension changer

A 1×1 convolution doesn't look at spatial neighbours. It's a per-pixel fully-connected layer across channels.

Uses:
- **Reduce channels** (256 → 64 before an expensive 3×3 conv) → saves computation
- **Add non-linearity** (1×1 conv + ReLU) without changing spatial size
- **Cross-channel mixing** — combine information across feature maps

In [ ]:
# Parameter counting practice
print("=== Parameter counting for a conv layer ===")
print("Formula: params = (k × k × C_in × C_out) + C_out")
print("                   ^^^^^^^^^^^^^^^^^^^^^^^^   ^^^^^")
print("                   kernel weights              biases")
print()

layers = [
    ("First conv", 3, 3, 3, 64),
    ("Standard conv", 3, 3, 64, 64),
    ("Upsample channels", 3, 3, 64, 128),
    ("1×1 bottleneck", 1, 1, 256, 64),
    ("1×1 expansion", 1, 1, 64, 256),
    ("Depthwise 3×3", 3, 3, 1, 128),  # depthwise: each channel separate
]

print(f"{'Layer':20s} {'Kernel':>8s} {'In→Out':>10s} {'Params':>10s}")
print("-" * 52)
for name, kh, kw, cin, cout in layers:
    params = kh * kw * cin * cout + cout
    print(f"{name:20s} {kh}×{kw:>5d} {cin:>4d}→{cout:<4d} {params:>10,}")

print("\n1×1 convs are MUCH cheaper than 3×3 convs.")
print("This is why bottleneck designs (ResNet, Inception) use 1×1 to reduce channels first.")

---
# Summary

| Concept | What | Why |
|---------|------|-----|
| **Convolution** | Slide a learned filter across the image | Detects features regardless of position (weight sharing) |
| **Multi-channel** | Kernel spans all input channels, produces one feature map | Combines information across channels |
| **Stride** | Skip positions when sliding | Downsamples the feature map |
| **Padding** | Add zeros around the border | Preserves spatial size |
| **Max pooling** | Take max in each window | Reduces size, adds translation invariance |
| **3×3 kernels** | Two 3×3 = one 5×5 receptive field | Fewer params + more non-linearity |
| **Skip connections** | output = F(x) + x | Gradient highway, enables 100+ layers |
| **1×1 convolution** | Per-pixel channel mixing | Reduces/expands channels cheaply |

**Output size formula:** $H_{out} = \lfloor(H_{in} + 2P - k) / S\rfloor + 1$

**Parameter formula:** $\text{params} = k \times k \times C_{in} \times C_{out} + C_{out}$

**Next:** The PyTorch CNN notebook — implement everything above and train on CIFAR-10.